[!["Open In Colab"](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ContextLab/llm-course/blob/main/slides/week7/gpt_from_scratch_demo.ipynb)

# Build GPT from scratch — step by step

**PSYC 51.17: Models of language and communication**
**Week 7**

---

## Learning objectives

By the end of this session, you will:
1. Implement a complete GPT model from scratch in PyTorch
2. Understand the role of each component (embeddings, attention, transformer blocks)
3. Train a mini-GPT on a small text corpus
4. Generate text using autoregressive decoding
5. Compare different sampling strategies (temperature, top-k, nucleus)

## Setup

In [ ]:
# Install required packages (for Colab)
!pip install -q torch tiktoken matplotlib numpy requests

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import tiktoken
import matplotlib.pyplot as plt
import numpy as np
import requests

print("\u2713 All imports successful!")

In [ ]:
# Download a small text corpus (Tiny Shakespeare)
import requests
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = requests.get(url).text
print(f"Downloaded {len(text):,} characters of Shakespeare")
print("First 200 characters:")
print("-" * 20)
print(text[:200])

## Part 1: Tokenization and data preparation

GPT models use **Byte-Pair Encoding (BPE)** to turn text into numbers. We'll use OpenAI's `tiktoken` library, which implements the same tokenizer used by GPT-2 and GPT-3. We also need to prepare our data in a "sliding window" format where each input sequence is paired with a target sequence shifted by one token.

In [ ]:
# Initialize the tokenizer
tokenizer = tiktoken.get_encoding("gpt2")

# Tokenize the entire text
tokens = tokenizer.encode(text)
print(f"Total tokens: {len(tokens):,}")

class TextDataset(Dataset):
    def __init__(self, tokens, max_seq_len):
        self.tokens = tokens
        self.max_seq_len = max_seq_len

    def __len__(self):
        return len(self.tokens) - self.max_seq_len

    def __getitem__(self, idx):
        # Extract a chunk of tokens of length max_seq_len + 1
        chunk = self.tokens[idx : idx + self.max_seq_len + 1]
        # Input (x) is the first max_seq_len tokens
        x = torch.tensor(chunk[:-1], dtype=torch.long)
        # Target (y) is the same chunk shifted by 1
        y = torch.tensor(chunk[1:], dtype=torch.long)
        return x, y

# Create a small dataset for demonstration
max_seq_len = 128
dataset = TextDataset(tokens, max_seq_len)

# Show an example of input/target offset
x, y = dataset[0]
print("\nExample input/target offset:")
print(f"Input (x):  {x[:10].tolist()}...")
print(f"Target (y): {y[:10].tolist()}...")
print("-" * 40)
print(f"Input text:  '{tokenizer.decode(x[:10].tolist())}'")
print(f"Target text: '{tokenizer.decode(y[:10].tolist())}'")

### 💡 Discussion

- Why do we shift the target by exactly one token?
- What happens if a word is not in the tokenizer's vocabulary?
- How does the `max_seq_len` affect what the model can learn?

## Part 2: Building the model

Now we'll implement the GPT architecture. This involves several components:
1. **Embeddings**: Combining token identity with positional information.
2. **Multi-Head Attention**: Allowing tokens to "look at" other tokens in the past.
3. **Feed-Forward Network**: A simple two-layer MLP applied to each token.
4. **Transformer Block**: Combining attention and feed-forward with residual connections and layer normalization.
5. **GPT**: The full stack of transformer blocks.

In [ ]:
class Embeddings(nn.Module):
    def __init__(self, vocab_size, d_model, max_seq_len, dropout):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(max_seq_len, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        seq_len = x.size(1)
        tok_emb = self.token_embed(x)                        # (B, T, D)
        pos_emb = self.pos_embed(torch.arange(seq_len, device=x.device))  # (T, D)
        return self.dropout(tok_emb + pos_emb)               # Broadcasting adds positions

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        self.q_linear = nn.Linear(d_model, d_model)
        self.k_linear = nn.Linear(d_model, d_model)
        self.v_linear = nn.Linear(d_model, d_model)
        self.out_linear = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        B, T, D = x.shape
        Q = self.q_linear(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        K = self.k_linear(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        V = self.v_linear(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        # Q, K, V shape: (B, n_heads, T, head_dim)

        # Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Apply attention to values and concatenate heads
        out = torch.matmul(attn_weights, V)                  # (B, n_heads, T, head_dim)
        out = out.transpose(1, 2).contiguous().view(B, T, D) # (B, T, D)
        return self.out_linear(out)

class FeedForward(nn.Module):
    def __init__(self, d_model, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),                        # GPT uses GELU, not ReLU
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(dropout)
        )
    def forward(self, x):
        return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attention = MultiHeadAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, dropout)

    def forward(self, x, mask):
        x = x + self.attention(self.ln1(x), mask)   # Pre-norm + residual
        x = x + self.ffn(self.ln2(x))               # Pre-norm + residual
        return x

def create_causal_mask(seq_len, device):
    return torch.tril(torch.ones(seq_len, seq_len, device=device))

class GPT(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, max_seq_len, dropout):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.embeddings = Embeddings(vocab_size, d_model, max_seq_len, dropout)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, dropout) for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, x, targets=None):
        seq_len = x.size(1)
        mask = create_causal_mask(seq_len, x.device)

        x = self.embeddings(x)
        for block in self.blocks:
            x = block(x, mask)
        x = self.ln_f(x)
        logits = self.lm_head(x)          # (batch, seq_len, vocab_size)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1)
            )
        return logits, loss

In [ ]:
# Model configuration (small for fast training)
config = {
    'vocab_size': 50257,    # GPT-2 vocabulary size
    'd_model': 128,         # Embedding dimension
    'n_layers': 4,          # Number of transformer blocks
    'n_heads': 4,           # Number of attention heads
    'max_seq_len': 128,     # Maximum sequence length
    'dropout': 0.1,
    'batch_size': 16,
    'learning_rate': 5e-4,
    'num_epochs': 3
}

device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = GPT(
    config['vocab_size'], config['d_model'], config['n_layers'],
    config['n_heads'], config['max_seq_len'], config['dropout']
).to(device)

# Print parameter count
params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {params:,}")

### 💡 Discussion

- Why do we use a causal mask in the attention layer?
- What is the purpose of the residual connections (`x = x + ...`)?
- How does the number of heads affect the model's ability to process information?

## Part 3: Training

We'll train our model using the **AdamW** optimizer, which is the standard for transformers. We'll also use **gradient clipping** to prevent the gradients from "exploding" and making training unstable.

In [ ]:
# Prepare data loader
dataloader = DataLoader(dataset, batch_size=config['batch_size'], shuffle=True)

optimizer = torch.optim.AdamW(
    model.parameters(), lr=config['learning_rate'],
    betas=(0.9, 0.95), weight_decay=0.1
)

model.train()
losses = []

for epoch in range(config['num_epochs']):
    total_loss = 0
    for i, (x, y) in enumerate(dataloader):
        # Only train on a subset of the data for the demo to keep it fast
        if i > 500: break 
        
        x, y = x.to(device), y.to(device)
        
        logits, loss = model(x, targets=y)
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item()
        if i % 100 == 0:
            print(f"Epoch {epoch+1}, Step {i}, Loss: {loss.item():.4f}")
            
    avg_loss = total_loss / min(len(dataloader), 501)
    losses.append(avg_loss)
    print(f"Epoch {epoch+1} complete. Average Loss: {avg_loss:.4f}")

In [ ]:
# Plot the training loss
plt.figure(figsize=(10, 6))
plt.plot(losses, marker='o', color='#00693e', linewidth=2)
plt.title("Training Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Cross Entropy Loss")
plt.grid(True, alpha=0.3)
plt.show()

### 💡 Discussion

- Why does the loss start high and decrease?
- What would happen if we trained for 100 epochs instead of 3?
- How does the batch size affect the speed and stability of training?

## Part 4: Text generation

Once trained, we can use the model to generate new text. This is done **autoregressively**: we give the model a prompt, it predicts the next token, we add that token to the prompt, and repeat. We'll implement several sampling strategies to control the "creativity" of the output.

In [ ]:
@torch.no_grad()
def generate_greedy(model, tokenizer, prompt, max_new_tokens=50):
    model.eval()
    tokens = tokenizer.encode(prompt)
    x = torch.tensor([tokens], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        x_crop = x[:, -model.max_seq_len:]
        logits, _ = model(x_crop)
        logits = logits[:, -1, :]                # Last position only
        next_token = torch.argmax(logits, dim=-1, keepdim=True)
        x = torch.cat([x, next_token], dim=1)

    return tokenizer.decode(x[0].tolist())

def sample_next_token(logits, temperature=1.0, top_k=None, top_p=None):
    logits = logits / temperature

    if top_k is not None:
        top_k = min(top_k, logits.size(-1))
        threshold = torch.topk(logits, top_k)[0][..., -1, None]
        logits[logits < threshold] = float('-inf')

    if top_p is not None:
        sorted_logits, sorted_idx = torch.sort(logits, descending=True)
        cumprobs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
        remove = cumprobs > top_p
        remove[..., 1:] = remove[..., :-1].clone()
        remove[..., 0] = False
        logits[sorted_idx[remove]] = float('-inf')

    probs = F.softmax(logits, dim=-1)
    return torch.multinomial(probs, num_samples=1)

@torch.no_grad()
def generate(model, tokenizer, prompt, max_new_tokens=100,
             temperature=0.8, top_k=40, top_p=0.9):
    model.eval()
    tokens = tokenizer.encode(prompt)
    x = torch.tensor([tokens], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        x_crop = x[:, -model.max_seq_len:]
        logits, _ = model(x_crop)
        next_token = sample_next_token(
            logits[0, -1, :], temperature=temperature, top_k=top_k, top_p=top_p
        )
        x = torch.cat([x, next_token.unsqueeze(0)], dim=1)

    return tokenizer.decode(x[0].tolist())

In [ ]:
prompt = "ROMEO:"
print(f"Prompt: {prompt}\n")

settings = [
    {"name": "Greedy", "func": lambda: generate_greedy(model, tokenizer, prompt)},
    {"name": "Creative (Temp=1.2)", "func": lambda: generate(model, tokenizer, prompt, temperature=1.2, top_k=None, top_p=None)},
    {"name": "Top-k (k=40)", "func": lambda: generate(model, tokenizer, prompt, temperature=1.0, top_k=40, top_p=None)},
    {"name": "Nucleus (p=0.9)", "func": lambda: generate(model, tokenizer, prompt, temperature=1.0, top_k=None, top_p=0.9)},
]

for s in settings:
    print(f"--- {s['name']} ---")
    print(s['func']())
    print()

### 💡 Discussion

- How does temperature change the "vibe" of the generated text?
- Why does greedy decoding often lead to repetitive loops?
- Which sampling strategy produced the most "Shakespeare-like" text?

## Part 5: Exploring what GPT learned

Even with a tiny model and short training time, GPT can pick up on patterns in the data. Let's see how it handles different prompts and extreme sampling settings.

In [ ]:
prompts = [
    "JULIET:",
    "The sun is",
    "To be, or not to be",
    "What is a"
]

print("Testing different prompts:\n")
for p in prompts:
    output = generate(model, tokenizer, p, max_new_tokens=30, temperature=0.8, top_k=40)
    print(f"Prompt: {p}")
    print(f"Output: {output}")
    print("-" * 20)

print("\nExtreme temperatures:")
print(f"Very low (0.1): {generate(model, tokenizer, 'ROMEO:', temperature=0.1, max_new_tokens=20)}")
print(f"Very high (5.0): {generate(model, tokenizer, 'ROMEO:', temperature=5.0, max_new_tokens=20)}")

### 💡 Discussion

- Does the model follow the formatting of the Shakespeare text (e.g., NAME:)?
- What happens when you give it a prompt that wasn't in the training data?
- How much better do you think the model would be with 10x more parameters and 10x more data?

## Summary

| Part | What we explored | Key insight |
|------|-----------------|-------------|
| Tokenization | tiktoken BPE | Text must be converted to sub-word units |
| Building GPT | Transformer architecture | Attention + Feed-Forward + Residuals = Power |
| Training | AdamW + Grad Clipping | Stable training requires careful optimization |
| Generation | Autoregressive decoding | Models predict one token at a time |
| Sampling | Temp, Top-k, Top-p | Randomness can be controlled for better quality |

## Further exploration

1. **Train longer**: Try increasing `num_epochs` to 10 or 20. How does the quality change?
2. **Bigger model**: Increase `d_model` to 256 or `n_layers` to 8. Watch the parameter count!
3. **New data**: Download a different text file (e.g., Project Gutenberg) and retrain.
4. **KV Caching**: Research how "Key-Value Caching" makes generation much faster in production.